# From Words to Embeddings - Concepts

**Business problem:** represent text so that a classifier or search tool can use it.

A model supports a decision; it does not make the business decision for us.

## Learning objectives

- Distinguish counts, weighted counts and learned word vectors.
- Calculate a dot product and cosine similarity.
- Explain what static word lookup and averaging lose.

## Dataset and practical scope

The short messages and numerical word vectors below are constructed examples, not measured model outputs.

All code is supplied. Run the cells from top to bottom.

## Libraries and settings

Use the prepared Python 3.11 Codespace kernel. No GPU or API key is needed.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 7
np.random.seed(SEED)
DATA_ROOT = Path('../../../00_Data')

## 1. A lexical baseline

TF-IDF weights word identities. Different spellings remain different coordinates. Vocabulary and word weights are learned from training text only.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
texts = ['claim prize', 'claim reward', 'meeting team']
vectorizer = TfidfVectorizer()
counts = vectorizer.fit_transform(texts)
display(pd.DataFrame(counts.toarray(), columns=vectorizer.get_feature_names_out()))
display(pd.DataFrame(cosine_similarity(counts), index=texts, columns=texts).round(3))

,claim,meeting,prize,reward,team
0,0.605349,0.000000,0.795961,0.000000,0.000000
1,0.605349,0.000000,0.000000,0.795961,0.000000
2,0.000000,0.707107,0.000000,0.000000,0.707107


,claim prize,claim reward,meeting team
claim prize,1.000,0.366,0.0
claim reward,0.366,1.000,0.0
meeting team,0.000,0.000,1.0


## 2. How word2vec learns

Skip-gram uses a centre word to predict nearby words. A learned input table supplies one word vector; a separate context table supplies prediction scores. Softmax turns scores into context-word probabilities. The loss uses an observed neighbouring word, not a spam or job-category label.

In [3]:
scores = np.array([2.0, 0.0, 1.0, -1.0, -2.0])
probabilities = np.exp(scores - scores.max())
probabilities /= probabilities.sum()
display(pd.DataFrame({'word': ['claim', 'prize', 'reward', 'meeting', 'team'],
                      'constructed score': scores, 'context probability': probabilities}))

,word,constructed score,context probability
0,claim,2.0,0.636409
1,prize,0.0,0.086129
2,reward,1.0,0.234122
3,meeting,-1.0,0.031685
4,team,-2.0,0.011656


## 3. Meaning as distance: use cosine carefully

A dot product multiplies matching entries and adds them. Cosine divides by the two vector lengths. It is not a probability, proof of synonymy or a causal explanation.

In [4]:
a, b = np.array([1.0, 2.0]), np.array([0.0, 1.0])
dot_product = a @ b
cosine = dot_product / (np.linalg.norm(a) * np.linalg.norm(b))
print('Dot product:', dot_product, '| Cosine:', round(cosine, 3))

Dot product: 2.0 | Cosine: 0.894


## 4. Average word vectors into a document

The following lookup table is invented. A mean of the same static word vectors ignores order.

In [5]:
word_vectors = {'bank': [1., 0.], 'paid': [0., 1.], 'supplier': [1., 1.]}
examples = ['bank paid supplier', 'supplier paid bank']
document_vectors = np.array([
    np.mean([word_vectors[word] for word in text.split()], axis=0) for text in examples])
display(pd.DataFrame(document_vectors, index=examples, columns=['value 1', 'value 2']))

,value 1,value 2
bank paid supplier,0.666667,0.666667
supplier paid bank,0.666667,0.666667


## Interpretation

Who pays whom in each example? Why can the static mean not recover that distinction? A contextual encoder can use positions before pooling, but does not guarantee correctness.

## Summary and business implications

Keep TF-IDF as a baseline. Word2vec learns related use, not guaranteed meaning. Train a separate classifier and test it on new documents.

## Limitations

This is a small teaching example, not evidence of deployment performance. Keep evaluation separate from fitting and model selection. Check errors and the user's decision, not only a headline metric.

## Further exercises

- Explain why shared contexts can also connect antonyms.
- What should happen when a message contains no known words?
- Complete the job-category exercise; its small synthetic dataset is not a business benchmark.

## References

**Required reading:** Stanford CS224N, Spring 2024, word-vector lecture, supplied excerpt.
**Instructor reference:** Mikolov et al. (2013), *Efficient Estimation of Word Representations in Vector Space*.
https://arxiv.org/abs/1301.3781
**Official documentation:** https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction

### Jupyter notebook - footer info

Include your name, submission date and a short description of your changes when submitting.